In [44]:
import cv2
import numpy as np
import time
from sympy import symbols, cos, sin, Matrix
from pymycobot.mycobot import MyCobot

START_POS = [0,0,0,0,0,0]
NEUTRAL = [0, -40, 55, 0, 30, 0]

bot = MyCobot('/dev/ttyACM0', 115200)

try:
    mtx = np.array(np.loadtxt('calibration/calib_mtx.csv', delimiter=','))
    dist = np.array(np.loadtxt('calibration/calib_dist.csv', delimiter=','))
except FileNotFoundError:
    print('Files not found, make sure you ran calibration!')

aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

# detector = cv2.aruco.ArucoDetector(aruco_dict)

tagSize = 38 / 1000 # m

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

def trans(a, alpha, d, theta):
    return Matrix([
        [cos(theta), -sin(theta), 0, a],
        [sin(theta)*cos(alpha), cos(theta)*cos(alpha), -sin(alpha), -sin(alpha)*d],
        [sin(theta)*sin(alpha), cos(theta)*sin(alpha), cos(alpha), cos(alpha)*d],
        [0, 0, 0, 1]
    ])

qs = symbols('q1:7')
q1, q2, q3, q4, q5, q6 = qs
l0, l1, l2, l3, l4, l5 = [114, 0, 100, 97, 0, 56] # mm
alpha2, alpha4, alpha5, alpha6 = np.radians([-90, -90, 90, -90]) # rad

l3_off = 10 #mm
j2_off = np.radians(-90) # rad

dh = [
    [0, 0, l0, q1],
    [0, alpha2, l1, q2 + j2_off],
    [l2, 0, 0, q3],
    [l3_off, alpha4, l3, q4],
    [0, alpha5, l4, q5],
    [0, alpha6, l5, q6]
]

T = Matrix.eye(4)
for row in dh:
    T *= trans(*row)

T_ec = np.array([
    [0, 0, 1, -40],
    [1, 0, 0, -60],
    [0, 1, 0, -15],
    [0, 0, 0, 1]
])

Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md


In [5]:
bot.send_angles(START_POS, 40)
time.sleep(2)
bot.set_gripper_value(95, 40, 1)
time.sleep(2)
bot.send_angles(NEUTRAL, 40)
time.sleep(2)

In [45]:
cv2.destroyAllWindows()
cv2.waitKey(1)

cap = cv2.VideoCapture(1)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    # corners, ids, _ = detector.detectMarkers(frame)
    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    key = cv2.waitKey(1)

    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            distance = np.linalg.norm(tvec) 
            t_rounded = np.round(tvec * 1000, 2).flatten().tolist()
            text = ", ".join(map(str, t_rounded))

            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.putText(frame, text, (int(square[0][0][0]), int(square[0][0][1]) + 20), cv2.FONT_HERSHEY_COMPLEX, 0.5, (255,255,255), 2)

            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)
    

    if key == ord(' '):
        bot_angles = bot.get_angles()
        joint_angles = [np.radians(j) for j in (bot_angles if bot_angles != -1 else NEUTRAL)]
        tvec_4 = np.concatenate((tvec.flatten() * 1000, [1]))
        T_sub = T.subs(zip([q1, q2, q3, q4, q5, q6], joint_angles)).evalf()
        obj = T_sub @ (T_ec @ tvec_4)
        ee_pos = np.concatenate((obj[:3], [-160, 90, -160]))
        print(ee_pos)

    cv2.imshow('ArUco', frame)

    if key == ord('r'):
        bot.set_gripper_value(95, 40, 1)
        time.sleep(2)
        bot.send_angles(NEUTRAL, 40)
        time.sleep(2)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

[202.465283276853 79.9017542802721 -30.9543923641296 -160 90 -160]
[127.735336533642 79.4032880061101 -95.5031809130853 -160 90 -160]
[199.307076786315 79.2511664668697 -28.1285995153910 -160 90 -160]
[205.736296445228 102.449593400048 22.9830970805904 -160 90 -160]
[233.195888308394 131.393558993385 -66.0122864152940 -160 90 -160]


-1